<a href="https://colab.research.google.com/github/Dev-007-Dev/ACT04-WEEK2/blob/main/ACT04_WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import getpass
import time
import json
import requests
from typing import List, Dict, Any

# ==========================================
# 1. API Configuration & Key Handling
# ==========================================
# Securely prompt for the OpenRouter API key without exposing it in plaintext
if "OPENROUTER_API_KEY" not in os.environ:
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API Key: ")

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# ==========================================
# 2. Core Model Invocation Function
# ==========================================
def query_model(
    model_name: str,
    prompt: str,
    system_prompt: str = "You are a helpful and precise AI assistant.",
    temperature: float = 0.7,
    max_tokens: int = 1000
) -> Dict[str, Any]:
    """
    Sends a chat completion request to OpenRouter for a specified model.
    Captures output, latency, and token usage metrics.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "GenAI Multibot Playground",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model_name,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    start_time = time.time()
    try:
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=60)
        elapsed_time = round(time.time() - start_time, 2)

        if response.status_code == 200:
            data = response.json()
            choice = data["choices"][0]["message"]["content"]
            usage = data.get("usage", {})
            return {
                "model": model_name,
                "status": "success",
                "latency_sec": elapsed_time,
                "prompt_tokens": usage.get("prompt_tokens", 0),
                "completion_tokens": usage.get("completion_tokens", 0),
                "total_tokens": usage.get("total_tokens", 0),
                "output": choice
            }
        else:
            return {
                "model": model_name,
                "status": f"error_{response.status_code}",
                "latency_sec": elapsed_time,
                "output": response.text
            }
    except Exception as e:
        return {
            "model": model_name,
            "status": "exception",
            "latency_sec": round(time.time() - start_time, 2),
            "output": str(e)
        }

# ==========================================
# 3. Comparative Multi-Model Playground Runner
# ==========================================
def run_comparative_playtest(
    task_name: str,
    prompt: str,
    models: List[str],
    temperature: float = 0.7
):
    """
    Runs the exact same prompt across multiple models and displays side-by-side metrics.
    """
    print(f"\n{'='*80}\nTASK: {task_name}\nPROMPT: {prompt}\n{'='*80}")

    results = []
    for model in models:
        print(f"--> Querying: {model} ...")
        res = query_model(model_name=model, prompt=prompt, temperature=temperature)
        results.append(res)

    for res in results:
        print(f"\n--- [Model: {res['model']}] | Status: {res['status']} | Latency: {res['latency_sec']}s ---")
        if res['status'] == 'success':
            print(f"Token Usage -> Prompt: {res['prompt_tokens']} | Completion: {res['completion_tokens']} | Total: {res['total_tokens']}")
            print("Response:")
            print(res['output'])
        else:
            print(f"Error details: {res['output']}")
        print("-" * 60)


Enter your OpenRouter API Key: ··········


In [ ]:
# Select models to compare across providers
TEST_MODELS = [
    "openai/gpt-4o-mini",
    "anthropic/claude-3.5-haiku",
    "meta-llama/llama-3.3-70b-instruct"
]

# Task 1: Creative Writing
creative_prompt = (
    "Write a short, evocative sci-fi story (max 150 words) about an ancient deep-space satellite "
    "that receives an unexpected transmission from Earth after 1,000 years of silence."
)
run_comparative_playtest("1. Creative Writing", creative_prompt, TEST_MODELS, temperature=0.8)

# Task 2: Analytical Reasoning
reasoning_prompt = (
    "A cloud architecture team has two options for hosting an agentic AI system:\n"
    "Option A: Multi-tenant serverless containers with dynamic scaling.\n"
    "Option B: Dedicated GPU instances with reserved capacity.\n"
    "Evaluate both options across (1) Token Latency/Cold Starts, (2) Cost Predictability at Scale, and (3) Hardware Optimization.\n"
    "Provide a final recommendation for a workload with unpredictable burst traffic."
)
run_comparative_playtest("2. Analytical Reasoning", reasoning_prompt, TEST_MODELS, temperature=0.2)

# Task 3: Instruction Following
instruction_prompt = (
    "Extract the project milestones from the following text into valid JSON format.\n"
    "The JSON must have a top-level key 'milestones', which is an array of objects containing keys: 'id' (integer), 'name' (string), 'owner' (string), and 'due_date' (YYYY-MM-DD).\n"
    "Output ONLY pure JSON. Do not include markdown code fences (```), commentary, or extra text.\n\n"
    "Text: 'Sprint Alpha kicks off with Core Engine Setup assigned to Alex due on 2026-10-15. Next, UI Wireframes will be handled by Priya by 2026-10-28. Finally, Security Audit must be finished by Marcus on 2026-11-12.'"
)
run_comparative_playtest("3. Instruction Following", instruction_prompt, TEST_MODELS, temperature=0.0)



TASK: 1. Creative Writing
PROMPT: Write a short, evocative sci-fi story (max 150 words) about an ancient deep-space satellite that receives an unexpected transmission from Earth after 1,000 years of silence.
--> Querying: openai/gpt-4o-mini ...
--> Querying: anthropic/claude-3.5-haiku ...
--> Querying: meta-llama/llama-3.3-70b-instruct ...

--- [Model: openai/gpt-4o-mini] | Status: success | Latency: 4.08s ---
Token Usage -> Prompt: 57 | Completion: 205 | Total: 262
Response:
In the cold darkness of the void, the ancient satellite, Vanguard-7, drifted through the remnants of a once-bustling solar system, its solar panels dulled by a millennium of cosmic dust. For a thousand years, it recorded the silence of the stars, its only companions the echoes of forgotten signals.

Then, one day, a flicker of light danced across its sensors. A transmission, weak but pulsing with urgency, crackled through the void. The haunting voice of a child called out, “Is anyone there?” 

Heartbeats reverber



## Comparative Evaluation and Architectural Analysis in Multi-LLM Systems

### 1. Task Performance and Comparative Suitability
In this evaluation, two primary model architectures were tested across three distinct operational domains: **OpenAI’s GPT-4o-mini** (a lightweight proprietary model) and **Meta’s Llama 3.3 70B Instruct** (an open-weight model).

* **Task 1: Creative Writing:** Both models effectively captured the melancholy of an ancient satellite receiving a transmission after a millennium. However, **`openai/gpt-4o-mini`** produced a more emotionally resonant narrative (*"Vanguard-7"*, the distant voice of a child, and the poetic contrast between cosmic dust and fading signals). In contrast, **`meta-llama/llama-3.3-70b-instruct`** (*"Aurora's Vigil"*) delivered a concise story with crisp prose (164 tokens vs. 205 tokens for GPT-4o-mini). While both satisfied the 150-word constraint, GPT-4o-mini demonstrated slightly better narrative pacing.
* **Task 2: Analytical Reasoning:** Both models thoroughly evaluated the trade-offs of serverless containers versus dedicated GPU instances across latency, cost predictability, and hardware customization. However, **`meta-llama/llama-3.3-70b-instruct`** provided a superior architectural analysis by introducing an industry-standard **hybrid architecture** (provisioning dedicated GPU capacity for baseline traffic while offloading unpredictable burst spikes to dynamically scaled serverless layers). GPT-4o-mini took a more rigid binary stance, recommending only dedicated instances.
* **Task 3: Instruction Following:** Both models achieved perfect zero-shot execution on structured JSON extraction. Crucially, both strictly obeyed the negative formatting constraints (*"Output ONLY pure JSON. Do not include markdown code fences or commentary"*), emitting clean, immediately parseable JSON objects without enclosing backticks.

---

### 2. Cross-Model Variance: Speed, Formatting, Tone, and Quality

| Dimension | `openai/gpt-4o-mini` | `meta-llama/llama-3.3-70b-instruct` |
| :--- | :--- | :--- |
| **Response Latency** | **Fast & Consistent** (1.69s – 9.41s) | **Variable** (1.23s on JSON; 28.79s on deep reasoning) |
| **Instruction Compliance** | Strict schema & zero code fences | Strict schema & zero code fences |
| **Tone & Style** | Conversational, narrative-rich, direct | Formal, structured, comprehensive systems engineering tone |
| **Token Efficiency** | Higher token output on narrative generation (205 tokens) | Higher token output on analytical depth (724 tokens) |

The most significant operational difference was latency during complex generation. While Llama 3.3 70B was faster on brief JSON extraction (1.23s vs. 1.69s), it required 28.79s to complete the deep analytical evaluation compared to 9.41s for GPT-4o-mini. This illustrates the trade-off between open-weight cloud-hosted inference latency and optimized proprietary hosting endpoints.

---

### 3. API Mechanics and Operational Parameters
The OpenRouter unified gateway abstracts provider differences into standard parameters:
* **Temperature Tuning:** A high temperature (`0.8`) on Creative Writing enriched figurative metaphors across both models, whereas setting temperature to `0.0` for Instruction Following eliminated stochastic drift, yielding identical, deterministic JSON schema structures.
* **Token Consumption Metrics:** Llama 3.3 70B processed slightly larger input token counts for the same prompts (170 prompt tokens vs. 164 for GPT-4o-mini in Task 3), illustrating tokenizer differences (Byte-Pair Encoding variations between OpenAI and Meta tokenizers).
* **Endpoint Routing & Error Handling:** The query to `anthropic/claude-3.5-haiku` produced an immediate `HTTP 404 No Endpoints Found` error (0.07s). This demonstrated the critical importance of defensive exception handling, dynamic fallbacks, and model slug validation in multi-LLM production code.

---

### 4. Designing an Automated Model Router
To balance latency, operational costs, and response quality, an enterprise system should employ an **Intelligent Intent-Based Model Router**:

1. **Lightweight Semantic Classifier:** A fast classification model (or small embedding model) inspects the prompt and classifies the task into *Data Extraction*, *Creative Ideation*, or *Complex Architectural Reasoning*.
2. **Deterministic Schema Gateway:** High-frequency, deterministic tasks (e.g., regex/JSON extraction) are routed to low-latency models like `gpt-4o-mini` or lightweight 8B/70B open-weight models with `temperature=0.0`.
3. **Deep Synthesis Pipeline:** When the router detects complex reasoning or system design prompts, it routes queries to higher-capacity reasoning models (such as `llama-3.3-70b` or Claude 3.5 Sonnet) with dynamic token allocations.
4. **Resilience & Circuit Breaking:** If an endpoint fails (such as the 404 or a 429 rate limit), the gateway catches the exception and reroutes the request to a fallback equivalent tier model with zero disruption to the user.

---

### 5. Strategic Strengths and Limitations of Multi-Provider Frameworks
Using multi-provider gateways provides significant strategic benefits:
* **Vendor Independence:** Applications avoid vendor lock-in, enabling dynamic cost optimization and leveraging market price drops across providers.
* **Specialization:** Workloads can be assigned to models based on their natural strengths (e.g., open-weight models for self-hosted compliance and private data; proprietary models for speed and managed uptime).

**Limitations & Challenges:**
* **Tokenizer & Prompt Divergence:** Subtle differences in how models interpret system prompts or formatting delimiters require additional sanitization layers.
* **Non-Uniform Latency Profiles:** As observed in Task 2, differing inference backends create latency variance (9s vs. 28s), complicating SLA guarantees in real-time user-facing applications.

---